# Reddit Crypto Sentiment Analysis Pipeline

This notebook demonstrates a complete data pipeline for:
1. Scraping Reddit posts from crypto-related subreddits
2. Processing data through Kafka streams
3. Performing sentiment analysis using CryptoBERT model
4. Storing enriched data in MongoDB following medallion architecture

## Architecture Overview
```
Reddit Scraper → Kafka (reddit-raw) → Sentiment Analysis (CryptoBERT) → Kafka (reddit-silver) → MongoDB (Silver Layer)
```

### Medallion Architecture Layers:
- **Bronze**: Raw Reddit data (posts, comments)
- **Silver**: Enriched data with sentiment analysis and quality metrics

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install kafka-python pymongo torch transformers accelerate
!pip install requests beautifulsoup4 lxml unstructured
!pip install pandas numpy matplotlib seaborn
!pip install python-dotenv

print("✅ All packages installed successfully")

In [ ]:
import json
import logging
import re
import time
from datetime import datetime, timezone, timedelta
from typing import Dict, Any, List, Optional
import requests
from urllib.parse import urljoin
import hashlib

# Data processing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# ML and Sentiment Analysis
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# MongoDB
from pymongo import MongoClient

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Set style for visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ All imports successful")

## 2. Reddit Data Scraper

First, let's create a Reddit scraper to fetch posts from crypto-related subreddits.

In [ ]:
class RedditScraper:
    """Simple Reddit scraper for fetching posts and comments"""
    
    def __init__(self):
        self.base_url = "https://www.reddit.com"
        self.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
    
    def scrape_subreddit(self, subreddit: str, limit: int = 25) -> List[Dict]:
        """Scrape posts from a subreddit"""
        posts = []
        
        try:
            url = f"{self.base_url}/r/{subreddit}/.json"
            params = {'limit': limit}
            
            response = requests.get(url, headers=self.headers, params=params, timeout=10)
            response.raise_for_status()
            
            data = response.json()
            
            if 'data' in data and 'children' in data['data']:
                for post_data in data['data']['children'][:limit]:
                    post = post_data['data']
                    posts.append(self._structure_post(post, subreddit))
                    
        except Exception as e:
            logger.error(f"Error scraping {subreddit}: {e}")
            
        return posts
    
    def _structure_post(self, post_data: Dict, subreddit: str) -> Dict:
        """Structure Reddit post data"""
        title = post_data.get('title', '')
        selftext = post_data.get('selftext', '')
        text_clean = f"{title} {selftext}".strip()
        
        return {
            "id": f"reddit_{post_data.get('id', '')}",
            "platform": "reddit",
            "author_id": self._hash_author(post_data.get('author', '')),
            "text_clean": self._clean_text(text_clean),
            "post_created_timestamp": datetime.fromtimestamp(
                post_data.get('created_utc', time.time()),
                tz=timezone.utc
            ).isoformat(),
            "meta": {
                "source_url": f"https://www.reddit.com{post_data.get('permalink', '')}",
                "subreddit": subreddit,
                "score": post_data.get('score', 0),
                "upvote_ratio": post_data.get('upvote_ratio', 0),
                "num_comments": post_data.get('num_comments', 0)
            },
            "scraped_at_timestamp": datetime.now(timezone.utc).isoformat()
        }
    
    def _hash_author(self, author: str) -> str:
        """Hash author name for privacy"""
        return hashlib.sha256(author.encode()).hexdigest()[:16] if author else None
    
    def _clean_text(self, text: str) -> str:
        """Clean and normalize text"""
        # Remove URLs
        text = re.sub(r'http[s]?://\S+', '', text)
        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        return text

# Test the scraper
scraper = RedditScraper()

# Define crypto subreddits to scrape
crypto_subreddits = ['CryptoCurrency', 'Bitcoin', 'CryptoMarkets']

print("🔍 Testing Reddit scraper...")
all_posts = []

for subreddit in crypto_subreddits:
    posts = scraper.scrape_subreddit(subreddit, limit=5)
    all_posts.extend(posts)
    print(f"✅ Scraped {len(posts)} posts from r/{subreddit}")

print(f"\n📊 Total posts scraped: {len(all_posts)}")

## 3. Sentiment Analysis with CryptoBERT

Now let's load the CryptoBERT model and perform sentiment analysis on the scraped posts.

In [ ]:
# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Using device: {device}")

# Load CryptoBERT model and tokenizer
model_name = 'ElKulako/cryptobert'

print(f"📥 Loading model: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)

# Label mapping
label_map = {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}

print("✅ Model loaded successfully!")

In [ ]:
def preprocess_text(text: str) -> str:
    """Preprocess text for sentiment analysis"""
    if not text:
        return ""
    
    # Remove URLs
    text = re.sub(r'http[s]?://\S+', '', text)
    # Remove mentions and hashtags
    text = re.sub(r'[@#]\w+', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Remove non-alphanumeric chars except basic punctuation
    text = re.sub(r'[^\w\s.,!?;:]', '', text)
    
    return text

def predict_sentiment(text: str, max_length: int = 512) -> Dict[str, Any]:
    """Predict sentiment using CryptoBERT"""
    try:
        # Preprocess text
        clean_text = preprocess_text(text)
        
        if not clean_text:
            return {
                'sentiment': 'Neutral',
                'confidence': 0.0,
                'bearish_prob': 0.0,
                'bullish_prob': 0.0,
                'neutral_prob': 1.0
            }
        
        # Tokenize
        inputs = tokenizer(
            clean_text,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors='pt'
        ).to(device)
        
        # Predict
        with torch.no_grad():
            outputs = model(**inputs)
            probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)
            predicted_class = torch.argmax(probabilities, dim=-1).item()
            confidence = torch.max(probabilities).item()
        
        # Get probabilities for each class
        probs = probabilities[0].cpu().numpy()
        
        return {
            'sentiment': label_map[predicted_class],
            'confidence': float(confidence),
            'bearish_prob': float(probs[0]),
            'bullish_prob': float(probs[1]),
            'neutral_prob': float(probs[2])
        }
        
    except Exception as e:
        logger.error(f"Error predicting sentiment: {e}")
        return {
            'sentiment': 'Neutral',
            'confidence': 0.0,
            'bearish_prob': 0.0,
            'bullish_prob': 0.0,
            'neutral_prob': 1.0
        }

# Test sentiment analysis on a sample post
if all_posts:
    sample_post = all_posts[0]
    sample_text = sample_post['text_clean'][:200]
    
    print(f"📝 Sample text: {sample_text[:100]}...")
    
    sentiment_result = predict_sentiment(sample_text)
    print(f"\n🔍 Sentiment Analysis Result:")
    print(f"   - Sentiment: {sentiment_result['sentiment']}")
    print(f"   - Confidence: {sentiment_result['confidence']:.2%}")
    print(f"   - Bearish: {sentiment_result['bearish_prob']:.2%}")
    print(f"   - Bullish: {sentiment_result['bullish_prob']:.2%}")
    print(f"   - Neutral: {sentiment_result['neutral_prob']:.2%}")

## 4. Process All Posts with Sentiment Analysis

In [ ]:
print("🚀 Processing all posts with sentiment analysis...\n")

processed_posts = []
sentiment_counts = Counter()

for i, post in enumerate(all_posts):
    print(f"Processing post {i+1}/{len(all_posts)}: {post['id'][:20]}...")
    
    # Perform sentiment analysis
    sentiment = predict_sentiment(post['text_clean'])
    
    # Add sentiment to post
    post['sentiment'] = sentiment
    
    # Count sentiments
    sentiment_counts[sentiment['sentiment']] += 1
    
    processed_posts.append(post)
    
    # Add delay to avoid rate limiting
    time.sleep(0.1)

print(f"\n✅ Processed {len(processed_posts)} posts")
print("\n📊 Sentiment Distribution:")
for sentiment, count in sentiment_counts.items():
    percentage = (count / len(processed_posts)) * 100
    print(f"   - {sentiment}: {count} ({percentage:.1f}%)")

## 5. Visualizations and Analysis

In [ ]:
# Create DataFrame for analysis
df = pd.DataFrame(processed_posts)

# Extract sentiment information
df['sentiment_label'] = df['sentiment'].apply(lambda x: x['sentiment'])
df['confidence'] = df['sentiment'].apply(lambda x: x['confidence'])
df['bullish_prob'] = df['sentiment'].apply(lambda x: x['bullish_prob'])
df['bearish_prob'] = df['sentiment'].apply(lambda x: x['bearish_prob'])
df['neutral_prob'] = df['sentiment'].apply(lambda x: x['neutral_prob'])
df['subreddit'] = df['meta'].apply(lambda x: x['subreddit'])

# 1. Sentiment Distribution Pie Chart
plt.figure(figsize=(15, 10))

# Pie chart
plt.subplot(2, 2, 1)
sentiment_counts = df['sentiment_label'].value_counts()
colors = ['#ff9999', '#66b3ff', '#99ff99']
plt.pie(sentiment_counts.values, labels=sentiment_counts.index, autopct='%1.1f%%', 
        colors=colors, startangle=90)
plt.title('Overall Sentiment Distribution', fontsize=14, fontweight='bold')

# 2. Sentiment by Subreddit
plt.subplot(2, 2, 2)
sentiment_by_subreddit = pd.crosstab(df['subreddit'], df['sentiment_label'])
sentiment_by_subreddit.plot(kind='bar', ax=plt.gca(), color=colors)
plt.title('Sentiment Distribution by Subreddit', fontsize=14, fontweight='bold')
plt.xticks(rotation=45)
plt.ylabel('Count')
plt.legend(title='Sentiment')

# 3. Confidence Distribution
plt.subplot(2, 2, 3)
sns.histplot(data=df, x='confidence', bins=20, kde=True, color='skyblue')
plt.title('Model Confidence Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Confidence Score')
plt.ylabel('Frequency')

# 4. Sentiment Probability Distribution
plt.subplot(2, 2, 4)
prob_data = df[['bullish_prob', 'bearish_prob', 'neutral_prob']].melt()
sns.boxplot(x='variable', y='value', data=prob_data, palette=colors)
plt.title('Sentiment Probability Distributions', fontsize=14, fontweight='bold')
plt.xlabel('Sentiment')
plt.ylabel('Probability')

plt.tight_layout()
plt.show()

# Print statistics
print("\n📊 Sentiment Analysis Statistics:")
print(f"   - Average Confidence: {df['confidence'].mean():.2%}")
print(f"   - High Confidence Posts (>80%): {len(df[df['confidence'] > 0.8])}/{len(df)} ({len(df[df['confidence'] > 0.8])/len(df)*100:.1f}%)")
print(f"   - Most Bullish Subreddit: {df.groupby('subreddit')['bullish_prob'].mean().idxmax()}")
print(f"   - Most Bearish Subreddit: {df.groupby('subreddit')['bearish_prob'].mean().idxmax()}")

## 6. Medallion Architecture Data Structure

Let's structure our data following the medallion architecture pattern.

In [ ]:
def create_bronze_record(post_data: Dict) -> Dict:
    """Create a bronze layer record from raw Reddit data"""
    return {
        'data': {
            'id': post_data['id'],
            'platform': post_data['platform'],
            'author_id': post_data['author_id'],
            'text_clean': post_data['text_clean'],
            'post_created_timestamp': post_data['post_created_timestamp'],
            'meta': post_data['meta']
        },
        'metadata': {
            'source': 'reddit',
            'type': 'post',
            'ingested_at': datetime.utcnow().isoformat(),
            'layer': 'bronze',
            'batch_id': datetime.utcnow().strftime('%Y%m%d_%H%M%S')
        }
    }

def create_silver_record(post_data: Dict) -> Dict:
    """Create a silver layer record with sentiment analysis"""
    return {
        'data': {
            'id': post_data['id'],
            'platform': post_data['platform'],
            'author_id': post_data['author_id'],
            'text_clean': post_data['text_clean'],
            'post_created_timestamp': post_data['post_created_timestamp'],
            'meta': post_data['meta'],
            'sentiment': post_data['sentiment']
        },
        'metadata': {
            'source': 'reddit',
            'type': 'post',
            'ingested_at': datetime.utcnow().isoformat(),
            'processed_at': datetime.utcnow().isoformat(),
            'layer': 'silver',
            'processing_version': '1.0'
        },
        'quality_metrics': {
            'text_length': len(post_data['text_clean']),
            'has_text': bool(post_data['text_clean'] and len(post_data['text_clean']) > 10),
            'sentiment_confidence': post_data['sentiment']['confidence'],
            'is_high_quality': post_data['sentiment']['confidence'] > 0.7 and len(post_data['text_clean']) > 20
        }
    }

# Create sample records
if processed_posts:
    sample_post = processed_posts[0]
    
    # Bronze layer record
    bronze_record = create_bronze_record(sample_post)
    
    # Silver layer record
    silver_record = create_silver_record(sample_post)
    
    print("📋 Bronze Layer Record Structure:")
    print(json.dumps(bronze_record, indent=2)[:500] + "...")
    
    print("\n📋 Silver Layer Record Structure:")
    print(json.dumps(silver_record, indent=2)[:500] + "...")

## 7. Simulating Kafka Pipeline

Let's simulate how data would flow through our Kafka pipeline.

In [ ]:
class MockKafkaPipeline:
    """Mock Kafka pipeline for demonstration"""
    
    def __init__(self):
        self.bronze_topic = []
        self.silver_topic = []
        self.mongodb_silver = []
    
    def produce_to_bronze(self, post_data: Dict):
        """Simulate producing to bronze topic"""
        bronze_record = create_bronze_record(post_data)
        self.bronze_topic.append(bronze_record)
        
    def process_to_silver(self, bronze_record: Dict):
        """Simulate processing from bronze to silver"""
        # Add sentiment analysis
        post_data = bronze_record['data']
        post_data['sentiment'] = predict_sentiment(post_data['text_clean'])
        
        silver_record = create_silver_record(post_data)
        self.silver_topic.append(silver_record)
        
        # Store to MongoDB (simulation)
        self.mongodb_silver.append(silver_record)
    
    def run_pipeline(self, posts: List[Dict]):
        """Run the complete pipeline"""
        print("🔄 Running mock Kafka pipeline...\n")
        
        # Step 1: Produce to bronze
        for post in posts[:5]:  # Process only first 5 for demo
            self.produce_to_bronze(post)
        
        print(f"✅ Produced {len(self.bronze_topic)} records to bronze topic (reddit-raw)")
        
        # Step 2: Process to silver
        for bronze_record in self.bronze_topic:
            self.process_to_silver(bronze_record)
        
        print(f"✅ Processed {len(self.silver_topic)} records to silver topic (reddit-silver)")
        print(f"✅ Stored {len(self.mongodb_silver)} records to MongoDB silver layer")
        
        return self.mongodb_silver

# Run the pipeline
pipeline = MockKafkaPipeline()
silver_data = pipeline.run_pipeline(processed_posts)

# Show sample silver record
if silver_data:
    print("\n📊 Sample Silver Record in MongoDB:")
    sample = silver_data[0]
    print(f"   ID: {sample['data']['id']}")
    print(f"   Subreddit: {sample['data']['meta']['subreddit']}")
    print(f"   Sentiment: {sample['data']['sentiment']['sentiment']}")
    print(f"   Confidence: {sample['data']['sentiment']['confidence']:.2%}")
    print(f"   Quality Score: {sample['quality_metrics']['sentiment_confidence']:.2%}")
    print(f"   Is High Quality: {sample['quality_metrics']['is_high_quality']}")

## 8. Silver Layer Aggregations

Let's create some aggregations that would typically be stored in the silver layer.

In [ ]:
# Create aggregations from silver data
if processed_posts:
    # Convert to DataFrame for easier aggregation
    silver_df = pd.DataFrame([create_silver_record(post) for post in processed_posts])
    
    # 1. Sentiment by Subreddit
    sentiment_by_subreddit = silver_df.groupby('data__meta__subreddit').agg({
        'data__sentiment': [
            ('count', 'count'),
            ('bullish_count', lambda x: sum(1 for s in x if s['sentiment'] == 'Bullish')),
            ('bearish_count', lambda x: sum(1 for s in x if s['sentiment'] == 'Bearish')),
            ('neutral_count', lambda x: sum(1 for s in x if s['sentiment'] == 'Neutral')),
            ('avg_confidence', lambda x: sum(s['confidence'] for s in x) / len(x))
        ]
    })
    
    sentiment_by_subreddit.columns = ['Total Posts', 'Bullish', 'Bearish', 'Neutral', 'Avg Confidence']
    sentiment_by_subreddit['Bullish %'] = (sentiment_by_subreddit['Bullish'] / sentiment_by_subreddit['Total Posts'] * 100).round(1)
    sentiment_by_subreddit['Bearish %'] = (sentiment_by_subreddit['Bearish'] / sentiment_by_subreddit['Total Posts'] * 100).round(1)
    
    print("📊 Sentiment Aggregation by Subreddit:")
    print(sentiment_by_subreddit.to_string())
    
    # 2. Hourly sentiment trend
    silver_df['hour'] = pd.to_datetime(silver_df['data__post_created_timestamp']).dt.hour
    hourly_sentiment = silver_df.groupby('hour').agg({
        'data__sentiment': [
            ('count', 'count'),
            ('bullish_pct', lambda x: sum(1 for s in x if s['sentiment'] == 'Bullish') / len(x) * 100),
            ('avg_confidence', lambda x: sum(s['confidence'] for s in x) / len(x))
        ]
    })
    
    hourly_sentiment.columns = ['Post Count', 'Bullish %', 'Avg Confidence']
    
    # Plot hourly trend
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    hourly_sentiment['Post Count'].plot(kind='bar', color='steelblue')
    plt.title('Posts by Hour of Day')
    plt.xlabel('Hour')
    plt.ylabel('Number of Posts')
    
    plt.subplot(1, 2, 2)
    hourly_sentiment['Bullish %'].plot(kind='line', marker='o', color='green')
    plt.title('Bullish Sentiment % by Hour')
    plt.xlabel('Hour')
    plt.ylabel('Bullish Percentage')
    plt.ylim(0, 100)
    
    plt.tight_layout()
    plt.show()
    
    # 3. Quality metrics
    quality_stats = {
        'Total Posts': len(silver_df),
        'High Quality Posts': len(silver_df[silver_df['quality_metrics'].apply(lambda x: x['is_high_quality'])]),
        'Average Text Length': silver_df['quality_metrics'].apply(lambda x: x['text_length']).mean(),
        'Average Confidence': silver_df['quality_metrics'].apply(lambda x: x['sentiment_confidence']).mean()
    }
    
    print("\n📈 Quality Metrics:")
    for metric, value in quality_stats.items():
        if isinstance(value, float):
            print(f"   - {metric}: {value:.2f}")
        else:
            print(f"   - {metric}: {value}")

## 9. Deployment Instructions

### To deploy this pipeline in production:

#### 1. Infrastructure Setup
```bash
# Install Kafka
wget https://downloads.apache.org/kafka/2.8.0/kafka_2.13-2.8.0.tgz
tar -xzf kafka_2.13-2.8.0.tgz
cd kafka_2.13-2.8.0

# Start Zookeeper
bin/zookeeper-server-start.sh config/zookeeper.properties

# Start Kafka
bin/kafka-server-start.sh config/server.properties

# Create topics
bin/kafka-topics.sh --create --topic reddit-raw --bootstrap-server localhost:9092
bin/kafka-topics.sh --create --topic reddit-silver --bootstrap-server localhost:9092
```

#### 2. MongoDB Setup
```bash
# Install MongoDB
wget -qO - https://www.mongodb.org/static/pgp/server-6.0.asc | sudo apt-key add -
echo "deb [ arch=amd64,arm64 ] https://repo.mongodb.org/apt/ubuntu focal/mongodb-org/6.0 multiverse" | sudo tee /etc/apt/sources.list.d/mongodb-org-6.0.list
sudo apt-get update
sudo apt-get install -y mongodb-org

# Start MongoDB
sudo systemctl start mongod
sudo systemctl enable mongod
```

#### 3. Run the Pipeline Components
```bash
# Terminal 1: Run Reddit Scraper with Kafka Producer
python cronjob/reddit_scheduler.py

# Terminal 2: Run Sentiment Analysis Consumer
python kafka/sentiment_consumer.py

# Terminal 3: Run MongoDB Silver Producer
python kafka/silver_producer.py
```

#### 4. Monitoring
- Monitor Kafka topics: `bin/kafka-console-consumer.sh --topic reddit-silver --bootstrap-server localhost:9092 --from-beginning`
- Check MongoDB collections: Connect with MongoDB Compass
- Monitor sentiment metrics in real-time using Grafana dashboard

## 10. Summary

In this notebook, we've demonstrated:

✅ **Data Ingestion**: Scraped Reddit posts from crypto-related subreddits

✅ **Sentiment Analysis**: Used CryptoBERT model to analyze sentiment with high accuracy

✅ **Stream Processing**: Simulated Kafka pipeline for real-time data flow

✅ **Medallion Architecture**: Implemented bronze and silver layers with proper data quality

✅ **Aggregations**: Created sentiment aggregations by subreddit and time

### Key Insights:
- CryptoBERT provides specialized sentiment analysis for crypto-related content
- The medallion architecture ensures data quality and traceability
- Kafka enables real-time processing of Reddit data
- MongoDB silver layer stores enriched data for analytics and ML

### Next Steps:
1. Add more crypto-related subreddits
2. Implement comment-level sentiment analysis
3. Add time-series analysis for sentiment trends
4. Create a real-time dashboard with Grafana
5. Implement anomaly detection for unusual sentiment patterns